# News Forecast Pipeline — 3 крупнейших СМИ РФ

**Цель:** Сбор → ETL → Анализ → Бэктест → Прогноз на **02.04.2026**

**СМИ:** Коммерсантъ, Лента.ру, Интерфакс

| Шаг | Ячейка | Описание |
|-----|--------|----------|
| 0 | Setup | Настройка окружения и импорты |
| 1 | Scrape | Сбор данных за 90 дней (RSS + архив) |
| 2 | ETL | Очистка, дедупликация, фильтрация |
| 3 | Analyze | Топики, частоты, NER, noise check |
| 4 | Backtest | Holdout-проверка на последних 7 днях |
| 5 | Metrics | Оценка качества прогноза |
| 6 | Forecast | Прогноз на целевую дату |
| 7 | Results | Просмотр итогового JSON |

---
## Ячейка 0 — Setup

In [ ]:
import sys, os, datetime
from dotenv import load_dotenv

# 1. Добавляем текущую директорию в путь импорта
sys.path.insert(0, os.getcwd())

# 2. Загружаем переменные окружения (.env)
load_dotenv()

import config, scraper, etl, analyzer, forecaster, backtester, metrics

print(f"Python: {sys.version.split()[0]}")
print(f"Рабочая директория: {os.getcwd()}")
print(f"Целевая дата прогноза: {config.TARGET_DATE}")

# Проверка наличия API ключа
if os.getenv('OPENROUTER_API_KEY'):
    print("✅ OpenRouter API Key: найден")
else:
    print("⚠️  OpenRouter API Key: НЕ НАЙДЕН (проверьте .env)")

---
## Ячейка 1 — Scrape: сбор данных

> ⏱️ **~5-30 минут** в зависимости от скорости соединения и количества доступных архивных страниц.
> 
> Можно ограничить список СМИ через `OUTLETS_TO_SCRAPE` или уменьшить `SCRAPE_FROM`.
> 
> Добавьте `enrich_leads=True` для полного скрапинга лидов (медленнее, по 1-2 сек/статья).

In [ ]:
import config, os, json
from scraper import scrape_all

# ── Настройки скрапинга ──────────────────────────────────────────
OUTLETS_TO_SCRAPE = config.OUTLET_SLUGS      # все 3, или например ["kommersant", "lenta"]
SCRAPE_FROM       = config.HISTORY_FROM       # дата начала (по умолчанию: TODAY − 90 дней)
SCRAPE_TO         = config.TODAY
ENRICH_LEADS      = False                    # True = доп. запросы за каждым лидом (медленно)
LOAD_EXISTING     = False                    # True = загрузить из JSON, если вы уверены, что он свежий
# ────────────────────────────────────────────────────────────────

if LOAD_EXISTING and os.path.exists(config.INTERMEDIATE_JSON):
    print(f"[pipeline] Загружаем существующий скрап из: {config.INTERMEDIATE_JSON}")
    with open(config.INTERMEDIATE_JSON, "r", encoding="utf-8") as f:
        scrape_results = json.load(f)
else:
    print("[pipeline] Файл не найден или LOAD_EXISTING=False. Запускаем скрапинг...")
    scrape_results = scrape_all(
        slugs=OUTLETS_TO_SCRAPE,
        start_date=SCRAPE_FROM,
        end_date=SCRAPE_TO,
        enrich_leads=ENRICH_LEADS,
    )

print("── Итог скрапинга ──")
for slug, recs in scrape_results.items():
    print(f"  {slug:<12} {len(recs):>5} записей")

---
## Ячейка 2 — ETL: очистка и дедупликация

In [ ]:
import importlib, etl as _etl_mod
importlib.reload(_etl_mod)
from etl import run_etl, load_clean
import pandas as pd

run_etl(slugs=config.OUTLET_SLUGS)

# Сводка по чистым данным
print("\n── Итог ETL ──")
summary_rows = []
for slug in config.OUTLET_SLUGS:
    df = load_clean(slug)
    if df.empty:
        summary_rows.append({"outlet": slug, "records": 0, "with_lead": 0,
                              "date_min": None, "date_max": None})
        continue
    df["published_at"] = pd.to_datetime(df["published_at"], errors="coerce")
    summary_rows.append({
        "outlet":   slug,
        "records":  len(df),
        "with_lead": df["lead"].notna().sum(),
        "date_min": df["published_at"].min().date() if not df.empty else None,
        "date_max": df["published_at"].max().date() if not df.empty else None,
    })

pd.DataFrame(summary_rows)


In [ ]:
# Просмотр нескольких строк для одного СМИ
PREVIEW_OUTLET = "kommersant"   # поменяйте на любой slug

df_preview = load_clean(PREVIEW_OUTLET)
df_preview["published_at"] = pd.to_datetime(df_preview["published_at"], errors="coerce")
df_preview.sort_values("published_at", ascending=False)[["published_at", "rubric", "title", "lead"]].head(10)

---
## Ячейка 3 — Analyze: темы, частоты, NER, noise check

In [ ]:
import config
from analyzer import analyze_all

analysis = analyze_all(slugs=config.OUTLET_SLUGS)


In [ ]:
# ── Топ-10 тем по каждому СМИ ────────────────────────────────────
import pandas as pd

TOPIC_OUTLET = "interfax"   # поменяйте на нужный slug

if TOPIC_OUTLET in analysis and analysis[TOPIC_OUTLET]:
    freq = analysis[TOPIC_OUTLET]["topic_freq"]
    print(f"\nТоп-10 тем ({config.OUTLETS[TOPIC_OUTLET]['name']}, последние {config.TOPIC_WINDOW} дней):\n")
    display(freq.head(10)[["cluster_name", "count", "pct"]])
else:
    print(f"Нет данных для {TOPIC_OUTLET} — сначала запустите Scrape+ETL")

In [ ]:
# ── Noise check: стабильность объёма по всем СМИ ─────────────────
noise_rows = []
for slug, res in analysis.items():
    if not res:
        continue
    n = res["noise"]
    noise_rows.append({
        "outlet":       slug,
        "daily_mean":   n.get("daily_mean"),
        "cv":           n.get("cv"),
        "stable":       "✅" if n.get("stable") else "⚠️",
        "top_rubric":   n.get("top_rubric"),
        "top_share_%":  round(n.get("top_rubric_share", 0) * 100, 1),
        "rubric_noisy": "⚠️" if n.get("rubric_noisy") else "✅",
    })

pd.DataFrame(noise_rows)

In [ ]:
# ── Топ сущностей (персоны, организации) ─────────────────────────
import plotly.graph_objects as go
from plotly.subplots import make_subplots

ENT_OUTLET = "kommersant"   # поменяйте
ENT_TYPE   = "persons"  # persons | orgs | locations

if ENT_OUTLET in analysis and analysis[ENT_OUTLET]:
    ents = analysis[ENT_OUTLET]["entities"].get(ENT_TYPE, pd.Series())
    if not ents.empty:
        fig = go.Figure(go.Bar(
            x=ents.values[:20][::-1],
            y=ents.index[:20][::-1],
            orientation="h"
        ))
        fig.update_layout(
            title=f"Топ упоминаний: {ENT_TYPE} — {config.OUTLETS[ENT_OUTLET]['name']}",
            height=500, margin=dict(l=200)
        )
        fig.show()
    else:
        print(f"Нет сущностей типа '{ENT_TYPE}' для {ENT_OUTLET}")

In [ ]:
# ── Динамика публикаций по дням (все СМИ) ────────────────────────
import plotly.express as px
from etl import load_all_clean

df_all = load_all_clean()
if not df_all.empty:
    daily = (
        df_all.groupby([df_all["published_at"].dt.date, "outlet"])
              .size()
              .reset_index(name="count")
    )
    daily.columns = ["date", "outlet", "count"]
    fig = px.line(daily, x="date", y="count", color="outlet",
                  title="Количество публикаций по дням",
                  labels={"count": "Публикаций", "date": "Дата"})
    fig.show()
else:
    print("Нет данных — запустите Scrape + ETL")

---
## Ячейка 4 — Backtest: holdout-проверка

> Обучение на истории до дня `D-1`, тест на каждом дне holdout.
> Методы: **inertia**, **frequency**, **calendar**, **llm**.
> Дни могут быть пропущены, если перед ними недостаточно исторических дней для честного окна частот.

In [ ]:
import config
from backtester import backtest_all

# ── Настройки ────────────────────────────────────────────────────
BT_OUTLETS     = config.OUTLET_SLUGS
BT_HOLDOUT     = config.BACKTEST_DAYS      # 7 дней
BT_METHODS     = ["inertia", "frequency", "calendar", "llm"]
# ────────────────────────────────────────────────────────────────

bt_results = backtest_all(
    slugs=BT_OUTLETS,
    holdout_days=BT_HOLDOUT,
    methods=BT_METHODS,
)

print("\n── Бэктест завершён ──")
for slug, days in bt_results.items():
    print(f"  {slug:<12} {len(days)} дней")


In [ ]:
# Просмотр одного дня бэктеста
import json

BT_OUTLET = "kommersant"
BT_DAY_IDX = 0    # индекс дня в holdout (0 = самый ранний)

if BT_OUTLET in bt_results and bt_results[BT_OUTLET]:
    day = bt_results[BT_OUTLET][BT_DAY_IDX]
    print(f"Дата: {day['date']}")
    print(f"\nФактические заголовки ({len(day['actual']['titles'])})")
    for t in day["actual"]["titles"][:10]:
        print(f"  ► {t}")
    print(f"\nПрогноз 'frequency' (топ-5 тем)")
    for p in day["predictions"].get("frequency", [])[:5]:
        print(f"  • {p.get('topic_label', '')[:80]}")
else:
    print(f"Нет данных бэктеста для {BT_OUTLET} — запустите ячейку выше")

---
## Ячейка 5 — Metrics: оценка качества

| Метрика | Целевое значение | Описание |
|---------|-----------------|----------|
| topic_hit_rate | ≥ 0.30 | Пересечение ключевых слов прогноза и реальных заголовков |
| entity_match_f1 | ≥ 0.20 | Совпадение персон и орг (F1) |
| semantic_similarity | ≥ 0.45 | Семантическая близость текстов (cosine) |
| style_match | ≥ 0.30 | Соответствие стилю СМИ |
| diversity_score | ≥ 0.70 | Разнообразие сгенерированных заголовков |

> Для методов без заголовков (`inertia`, `frequency`) text-level метрики показываются как `n/a`.

In [ ]:
import config
from metrics import evaluate_all

metrics_reports = evaluate_all(
    slugs=config.OUTLET_SLUGS,
)


In [ ]:
# ── Сводная таблица метрик ────────────────────────────────────────
import pandas as pd

TARGETS = {
    "topic_hit_rate":      0.30,
    "entity_match_f1":     0.20,
    "semantic_similarity": 0.45,
    "style_match":         0.30,
    "diversity_score":     0.70,
}

rows = []
for slug, rep in metrics_reports.items():
    row = {"outlet": slug, "method": rep.get("method")}
    for metric, target in TARGETS.items():
        val = rep.get(metric, 0)
        row[metric] = f"{val:.3f}  {'✅' if val >= target else '❌'}"
    rows.append(row)

pd.set_option("display.max_colwidth", 20)
pd.DataFrame(rows).set_index("outlet")

In [ ]:
# ── Визуализация метрик (radar chart) ────────────────────────────
import plotly.graph_objects as go

metric_cols = list(TARGETS.keys())
fig = go.Figure()

for slug, rep in metrics_reports.items():
    values = [rep.get(m, 0) for m in metric_cols]
    values_closed = values + [values[0]]  # close polygon
    cats_closed   = metric_cols + [metric_cols[0]]
    fig.add_trace(go.Scatterpolar(
        r=values_closed,
        theta=cats_closed,
        fill="toself",
        name=f"{config.OUTLETS[rep.get('outlet', slug)]['name']} ({rep.get('method', '?')})",
    ))

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    title="Метрики качества прогноза по СМИ",
    showlegend=True,
)
fig.show()

---
## Ячейка 6 — Forecast: прогноз на 02.04.2026

In [ ]:
import config, datetime
from forecaster import forecast_all

# ── Настройки ────────────────────────────────────────────────────
TARGET_DATE    = datetime.date(2026, 4, 2)
USE_LLM        = True   # False — пропустить генерацию заголовков через OpenRouter
FORECAST_SLUGS = config.OUTLET_SLUGS
# ────────────────────────────────────────────────────────────────

forecast_reports = forecast_all(
    slugs=FORECAST_SLUGS,
    target_date=TARGET_DATE,
    use_llm=USE_LLM,
)


In [ ]:
# ── Сводка по каждому СМИ ────────────────────────────────────────
for slug, rep in forecast_reports.items():
    if not rep:
        continue
    name   = rep.get("outlet_name", slug)
    preds  = rep.get("predictions", [])
    llm    = [p for p in preds if p.get("method") == "llm"]
    cal    = [p for p in preds if p.get("method") == "calendar"]
    freq   = [p for p in preds if p.get("method") == "frequency"]
    iner   = [p for p in preds if p.get("method") == "inertia"]

    print(f"\n{'='*60}")
    print(f"  {name} ({slug})")
    print(f"{'='*60}")
    print(f"  Топ-темы: {', '.join(rep.get('top_topics', [])[:3])}")
    print(f"  Контекст событий:\n    {rep.get('events_context', '').replace(chr(10), chr(10)+'    ')}")
    print(f"\n  Baseline — Инерция ({len(iner)} тем):")
    for p in iner[:5]:
        print(f"    · {p.get('topic_label', '')[:70]}")
    print(f"\n  Baseline — Частотность ({len(freq)} тем):")
    for p in freq[:5]:
        print(f"    · {p.get('topic_label', '')[:70]}")
    print(f"\n  Календарь событий ({len(cal)}):")
    for p in cal:
        print(f"    • {p.get('title', '')}")
    if llm:
        print(f"\n  LLM-заголовки ({len(llm)}):")
        for p in llm[:8]:
            print(f"    ▶ {p.get('title', '')}")
            lead = p.get("lead")
            if lead:
                print(f"      {lead[:120]}")

---
## Ячейка 7 — Results: просмотр итогового JSON

In [ ]:
import json, os
from IPython.display import JSON

json_path = os.path.join(config.FORECASTS_DIR, "forecast_2026-04-02.json")

if os.path.exists(json_path):
    with open(json_path, encoding="utf-8") as f:
        forecast_json = json.load(f)
    print(f"Файл: {json_path}")
    print(f"СМИ в файле: {list(forecast_json.keys())}")
    # Интерактивный JSON-просмотр в Jupyter
    JSON(forecast_json)
else:
    print(f"Файл не найден: {json_path}")
    print("Сначала запустите ячейку Forecast (шаг 6).")

In [ ]:
# ── Таблица всех LLM-прогнозов ────────────────────────────────────
import pandas as pd

rows = []
for slug, rep in forecast_reports.items():
    if not rep:
        continue
    for p in rep.get("predictions", []):
        if p.get("method") == "llm":
            rows.append({
                "СМИ":     config.OUTLETS[slug]["name"],
                "Рубрика": p.get("rubric", ""),
                "Заголовок": p.get("title", ""),
                "Лид":     (p.get("lead") or "")[:120],
            })

if rows:
    df_llm = pd.DataFrame(rows)
    pd.set_option("display.max_colwidth", 80)
    display(df_llm)
else:
    print("LLM-прогнозы не сгенерированы. Проверьте Ollama или запустите с USE_LLM=True.")

In [ ]:
# ── Экспорт прогнозов в Excel ─────────────────────────────────────
import config, datetime, os
import pandas as pd

TARGET_DATE = datetime.date(2026, 4, 2)
EXPORT_PATH = os.path.join(config.FORECASTS_DIR, f"forecast_{TARGET_DATE}.xlsx")

rows = []
for slug, rep in forecast_reports.items():
    if not rep:
        continue
    outlet_name = config.OUTLETS[slug]["name"]
    for p in rep.get("predictions", []):
        rows.append({
            "СМИ":         outlet_name,
            "Метод":       p.get("method", ""),
            "Рубрика":     p.get("rubric", "") or p.get("topic_label", ""),
            "Заголовок":   p.get("title", "") or p.get("topic_label", ""),
            "Лид":         (p.get("lead") or ""),
            "Уверенность": p.get("score", ""),
        })

if rows:
    df_export = pd.DataFrame(rows)

    with pd.ExcelWriter(EXPORT_PATH, engine="openpyxl") as writer:
        # Лист 1: все прогнозы
        df_export.to_excel(writer, sheet_name="Все прогнозы", index=False)

        # Лист 2: только LLM-заголовки
        df_llm = df_export[df_export["Метод"] == "llm"]
        if not df_llm.empty:
            df_llm.to_excel(writer, sheet_name="LLM заголовки", index=False)

        # Лист 3+: по одному листу на СМИ
        for slug, rep in forecast_reports.items():
            if not rep:
                continue
            name = config.OUTLETS[slug]["name"]
            df_outlet = df_export[df_export["СМИ"] == name]
            df_outlet.to_excel(writer, sheet_name=slug[:31], index=False)

    print(f"✅ Файл сохранён: {EXPORT_PATH}")
    print(f"   Строк: {len(df_export)}  |  Листов: {2 + len(forecast_reports)}")
else:
    print("⚠️  Нет данных для экспорта — сначала запустите ячейку Forecast.")


In [ ]:
# ── Таблица календарных событий на дату ──────────────────────────
from event_calendar import load_events

events = load_events(target_date=datetime.date(2026, 4, 2), window_days=3)
pd.DataFrame(events)[["date", "event_type", "description", "outlets"]]

---
## Быстрый запуск всего пайплайна одной ячейкой

In [ ]:
# !! Выполнит весь пайплайн последовательно !!
# Раскомментируйте нужный вариант
import subprocess, sys

# ВАРИАНТ 1: только прогноз без LLM (быстро, если данные уже собраны)
# subprocess.run([sys.executable, "main.py", "--mode", "forecast",
#                 "--target", "2026-04-02", "--no-llm"], check=True)

# ВАРИАНТ 2: полный пайплайн с LLM (~20-60 мин, Kommersant архив самый долгий)
#subprocess.run([sys.executable, "main.py", "--mode", "all",
#                  "--target", "2026-04-02"], check=True)

# ВАРИАНТ 3: только для одного СМИ
# subprocess.run([sys.executable, "main.py", "--mode", "all",
#                 "--outlets", "kommersant", "--target", "2026-04-02"], check=True)

print("Раскомментируйте нужный вариант выше и запустите ячейку.")
